# 🏢 HAVEN PROPTECH — NOTEBOOK HUẤN LUYỆN LOCAL SLM (QWEN2.5-0.5B LORA)

> **Dự án**: Nền tảng Cho Thuê & Quản Lý Bất Động Sản Tích Hợp AI (HAVEN Sanctuary Living)  
> **Môn học**: Ứng dụng Trí tuệ Nhân tạo trong Phát triển Phần mềm (KHMT K23A)  
> **Mục tiêu**: Tinh chỉnh (Fine-tune) mô hình ngôn ngữ nhỏ **Qwen2.5-0.5B-Instruct** trên dữ liệu 536 căn hộ thực tế của HAVEN bằng kỹ thuật **QLoRA** trên GPU T4 Google Colab, sau đó lượng tử hóa (Quantization) xuất ra file **GGUF Q4_K_M** (~350MB) để chạy độc lập Offline 100% trên máy cá nhân.

## 1. Cài đặt Thư Viện Unsloth & Tối Ưu Hóa GPU T4 (Miễn Phí)

In [ ]:
# Cài đặt Unsloth & các dependencies để tăng tốc huấn luyện x2 và giảm 70% VRAM
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets

## 2. Kết Nối Google Drive & Giải Nén Bộ Dữ Liệu HAVEN (1053 Mẫu Q&A)

In [ ]:
from google.colab import drive
import os
import zipfile

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/COLAB/01_haven_qa_dataset.zip'
EXTRACT_DIR = '/content/dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("✅ Đã giải nén thành công bộ dữ liệu HAVEN!")
    !head -n 2 /content/dataset/01_haven_qa_dataset.jsonl
else:
    print("❌ Không tìm thấy file zip trên Google Drive. Vui lòng kiểm tra đường dẫn:", ZIP_PATH)

## 3. Tải Mô Hình Nền Tảng (Base Model): Qwen2.5-0.5B-Instruct 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Tự động phát hiện (Float16 cho T4)
load_in_4bit = True # Lượng tử hóa 4-bit giúp tiết kiệm tối đa VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Đã nạp thành công Base Model Qwen2.5-0.5B!")

## 4. Thiết Lập Tham Số LoRA (Low-Rank Adaptation)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Cấu hình LoRA Adapter hoàn tất! Chuẩn bị nạp dữ liệu.")

## 5. Nạp Dataset & Chuẩn Hóa Theo Định Dạng ChatML

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="/content/dataset/01_haven_qa_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print("✅ Đã chuẩn hóa", len(dataset), "mẫu đối thoại ChatML!")

## 6. Huấn Luyện Tinh Chỉnh Mô Hình (Training với SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120, # Khoảng 2-3 epochs, mất 8-12 phút trên T4
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Bắt đầu quá trình huấn luyện LoRA...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện hoàn tất! Thông số thống kê:", trainer_stats)

## 7. Đánh Giá & Thử Nghiệm Suy Luận Cục Bộ (Local Inference Test)

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Bạn là HAVEN AI - Trợ lý Trí tuệ Nhân tạo Độc quyền của Nền tảng PropTech HAVEN (Sanctuary Living)."},
    {"role": "user", "content": "Tôi nuôi chó và thích chạy bộ quanh hồ, ngân sách tầm 18 triệu tại Tây Hồ, hãy tư vấn cho tôi?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.batch_decode(outputs)
print("\n--- KẾT QUẢ PHẢN HỒI CỦA HAVEN LOCAL AI ---\n")
print(response[0].split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())

## 8. Hợp Nhất Trọng Số (Merge LoRA) & Xuất File GGUF Q4_K_M Vào Google Drive

In [ ]:
# Xuất trực tiếp sang định dạng GGUF (lượng tử hóa 4-bit siêu nhẹ ~350MB)
# File này có thể mang về máy tính cá nhân để chạy với Ollama hoặc llama.cpp mà KHÔNG cần card đồ họa rời!

SAVE_GGUF_PATH = "/content/drive/MyDrive/COLAB/haven-qwen-0.5b-q4_k_m"
print("⏳ Đang lượng tử hóa và xuất file GGUF vào Google Drive...")
model.save_pretrained_gguf(SAVE_GGUF_PATH, tokenizer, quantization_method = "q4_k_m")
print("✅ ĐÃ XUẤT THÀNH CÔNG! File GGUF đã nằm tại Google Drive /COLAB!")

## 9. Hướng Dẫn Chạy Cục Bộ 1-Click Trên Máy Tính Cá Nhân Với Ollama

1. Tải file `haven-qwen-0.5b-q4_k_m.gguf` từ thư mục `G:\My Drive\COLAB` về thư mục dự án `D:\HAVEN`.
2. Tạo một file tên là `Modelfile` với nội dung:
   ```dockerfile
   FROM ./haven-qwen-0.5b-q4_k_m.gguf
   SYSTEM """Bạn là HAVEN AI - Trợ lý Bất động sản Thông minh của nền tảng HAVEN PropTech."""
   PARAMETER temperature 0.7
   PARAMETER top_p 0.9
   ```
3. Mở Terminal / PowerShell và khởi tạo mô hình nội bộ:
   ```powershell
   ollama create haven-ai -f Modelfile
   ollama run haven-ai
   ```
4. **Kiểm tra**: Mô hình sẽ suy luận mượt mà ngay trên CPU máy tính của bạn với tốc độ 30-50 tokens/giây mà không tốn chi phí gọi API bên thứ ba!